# 🌾 KHÁM PHÁ VÀ PHÂN TÍCH DỮ LIỆU TỪNG BƯỚC (STEP-BY-STEP EDA)
### Chuyên đề: Dữ liệu Lúa Gạo & Các biến vĩ mô nông nghiệp
**Mục tiêu:** Thực hiện EDA khám phá chi tiết từng bảng dữ liệu qua từng bước cụ thể:
- Kiểm tra cấu trúc (`.info()`, `.shape`, `.dtypes`)
- Kiểm tra mẫu dữ liệu (`.head()`, `.tail()`)
- Thống kê mô tả & dải giá trị (`.describe()`, min, max, mean, median)
- Đếm giá trị duy nhất & phân bố danh mục (`.nunique()`, `.unique()`, `.value_counts()`)
- Kiểm tra dữ liệu khuyết thiếu (`.isna().sum()`, `% missing`)
- Lọc và đào sâu riêng cho thị trường Việt Nam & ngành lúa gạo
- Phát hiện bất thường (giá bằng 0, đơn vị lệch, ký hiệu thiếu, ban hành xuất khẩu)


## 🛠️ Bước 0: Thiết lập Môi trường & Đường dẫn Dữ liệu


In [1]:
import os
import sys
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl

# Cấu hình hiển thị của Pandas để xem rõ dữ liệu
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) >= 100 else f'{x:.4g}')

# Tự động định vị thư mục gốc của dữ liệu raw
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "faostat").exists():
    DATA_DIR = CURRENT_DIR
elif (CURRENT_DIR / "raw" / "faostat").exists():
    DATA_DIR = CURRENT_DIR / "raw"
else:
    DATA_DIR = CURRENT_DIR

print(f"✅ Thư mục dữ liệu gốc: {DATA_DIR.resolve()}")
print(f"Các thư mục con: {[p.name for p in DATA_DIR.iterdir() if p.is_dir()]}")


✅ Thư mục dữ liệu gốc: C:\Users\hoang\source\repos\final term\tieu luan chuyen nganh\data_main\raw
Các thư mục con: ['faostat', 'nso', 'thitruongnongsan', 'usda', 'world_bank']


---
# 🌐 PHẦN 1: DỮ LIỆU TỔ CHỨC LƯƠNG NÔNG LIÊN HIỆP QUỐC (FAOSTAT)
Bao gồm:
1. `production_world.csv`: Diện tích, năng suất, sản lượng lúa thế giới (1961–2024).
2. `price.csv`: Giá nông sản tại cổng trại (Producer Prices).
3. `supply_utilization.csv`: Bảng cân đối cung - cầu lúa gạo toàn cầu (2010–2023).
4. `Trade_DetailedTradeMatrix...`: Ma trận thương mại xuất nhập khẩu gạo song phương.


### 1.1 Khám phá `production_world.csv` (Sản lượng lúa thế giới)


In [2]:
# Đọc 5 dòng đầu tiên để xem cấu trúc tổng quát
f_prod = DATA_DIR / "faostat" / "production_world.csv"
df_prod = pd.read_csv(f_prod)
print(f"Kích thước DataFrame: {df_prod.shape[0]:,} dòng x {df_prod.shape[1]} cột")
df_prod.head(5)


Kích thước DataFrame: 22,738 dòng x 15 cột


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,4,Afghanistan,5312,Area harvested,113,Rice,1961,1961,ha,"210,000.00",A,Official value,NaN
1,QCL,Crops and livestock products,4,Afghanistan,5412,Yield,113,Rice,1961,1961,kg/ha,"1,519.00",A,Official value,NaN
2,QCL,Crops and livestock products,4,Afghanistan,5510,Production,113,Rice,1961,1961,t,"319,000.00",A,Official value,NaN
3,QCL,Crops and livestock products,4,Afghanistan,5312,Area harvested,113,Rice,1962,1962,ha,"210,000.00",A,Official value,NaN
4,QCL,Crops and livestock products,4,Afghanistan,5412,Yield,113,Rice,1962,1962,kg/ha,"1,519.00",A,Official value,NaN


In [5]:
# Kiểm tra kiểu dữ liệu và bộ nhớ bằng .info()
df_prod.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22738 entries, 0 to 22737
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       22738 non-null  object 
 1   Domain            22738 non-null  object 
 2   Area Code (M49)   22738 non-null  int64  
 3   Area              22738 non-null  object 
 4   Element Code      22738 non-null  int64  
 5   Element           22738 non-null  object 
 6   Item Code (CPC)   22738 non-null  int64  
 7   Item              22738 non-null  object 
 8   Year Code         22738 non-null  int64  
 9   Year              22738 non-null  int64  
 10  Unit              22738 non-null  object 
 11  Value             22509 non-null  float64
 12  Flag              22738 non-null  object 
 13  Flag Description  22738 non-null  object 
 14  Note              389 non-null    object 
dtypes: float64(1), int64(5), object(9)
memory usage: 2.6+ MB


In [6]:
# Kiểm tra số lượng giá trị thiếu (Null/NaN) trên từng cột
missing_prod = pd.DataFrame({
    'Null Count': df_prod.isna().sum(),
    'Null %': (df_prod.isna().sum() / len(df_prod) * 100).round(2)
})
missing_prod


,Null Count,Null %
Domain Code,0,0
Domain,0,0
Area Code (M49),0,0
Area,0,0
Element Code,0,0
Element,0,0
Item Code (CPC),0,0
Item,0,0
Year Code,0,0
Year,0,0


In [7]:
# Khám phá các danh mục: Mặt hàng (Item), Chỉ tiêu (Element), Giai đoạn (Year)
print("Các mặt hàng có trong file:", df_prod['Item'].unique())
print("Mã CPC mặt hàng:", df_prod['Item Code (CPC)'].unique())
print("Các chỉ tiêu đo lường (Element):", df_prod['Element'].unique())
print("Đơn vị tương ứng (Unit):", df_prod['Unit'].unique())
print(f"Giai đoạn thời gian: Từ năm {df_prod['Year'].min()} đến năm {df_prod['Year'].max()} ({df_prod['Year'].nunique()} năm)")
print(f"Số lượng quốc gia/vùng lãnh thổ: {df_prod['Area'].nunique()} nước")


Các mặt hàng có trong file: ['Rice']
Mã CPC mặt hàng: [113]
Các chỉ tiêu đo lường (Element): ['Area harvested' 'Yield' 'Production']
Đơn vị tương ứng (Unit): ['ha' 'kg/ha' 't']
Giai đoạn thời gian: Từ năm 1961 đến năm 2024 (64 năm)
Số lượng quốc gia/vùng lãnh thổ: 146 nước


In [8]:
# Thống kê mô tả số học (min, max, mean, quantiles) cho cột giá trị đo lường 'Value'
df_prod.groupby('Element')['Value'].describe()


,count,mean,std,min,25%,50%,75%,max
Element,,,,,,,,
Area harvested,"7,599.00","1,507,233.79","5,717,269.34",0,"7,000.00","51,300.00","350,000.00","50,511,930.00"
Production,"7,599.00","5,791,096.44","25,028,846.49",0,"15,000.00","131,576.89","1,058,281.00","217,867,864.16"
Yield,"7,311.00","3,168.29","1,886.99",141.70,"1,725.70","2,777.80","4,251.00","13,385.80"


In [9]:
# Đào sâu dữ liệu riêng cho VIỆT NAM (Area Code M49 = 704 hoặc Area = 'Viet Nam')
df_vn_prod = df_prod[df_prod['Area'] == 'Viet Nam']
print(f"Tổng số dòng của Việt Nam: {len(df_vn_prod)} dòng")
print("Số năm theo dõi:", df_vn_prod['Year'].nunique(), "năm (1961 - 2024)")

# Pivot để xem chuỗi thời gian của Việt Nam qua các năm gần đây (2015 - 2024)
vn_pivot = df_vn_prod[df_vn_prod['Year'] >= 2015].pivot(index='Year', columns='Element', values='Value')
vn_pivot


Tổng số dòng của Việt Nam: 192 dòng
Số năm theo dõi: 64 năm (1961 - 2024)


Element,Area harvested,Production,Yield
Year,,,
2015,"7,828,607.00","45,090,561.89","5,759.70"
2016,"7,734,722.00","43,112,009.81","5,573.80"
2017,"7,708,534.00","42,763,682.38","5,547.60"
2018,"7,570,741.00","44,046,249.56","5,818.00"
2019,"7,451,544.00","43,495,487.58","5,837.10"
2020,"7,222,403.00","42,764,999.76","5,921.20"
2021,"7,219,797.00","43,852,728.69","6,074.00"
2022,"7,088,853.00","42,660,798.11","6,018.00"
2023,"7,115,058.00","43,497,624.81","6,113.50"


In [10]:
# Kiểm tra phân bố các cờ chất lượng (Flags) của FAO
print("Phân bố cờ dữ liệu (Flag) trên toàn cầu:")
print(df_prod['Flag Description'].value_counts())


Phân bố cờ dữ liệu (Flag) trên toàn cầu:
Flag Description
Official value                         18566
Estimated value                         2787
Value imputed by a receiving agency      673
Value from external organization         483
Missing value; data cannot exist         229
Name: count, dtype: int64


### 1.2 Khám phá `price.csv` (Giá sản xuất tại cổng trại - Producer Prices)


In [11]:
# Đọc dữ liệu giá nông dân bán tại cổng trại
f_price = DATA_DIR / "faostat" / "price.csv"
df_price = pd.read_csv(f_price)
print(f"Kích thước bảng giá: {df_price.shape[0]:,} dòng x {df_price.shape[1]} cột")
df_price.head(5)


Kích thước bảng giá: 15,334 dòng x 16 cột


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Months Code,Months,Unit,Value,Flag,Flag Description
0,PP,Producer Prices,4,Afghanistan,5530,Producer Price (LCU/tonne),113,Rice,1993,1993,7021,Annual value,LCU,"6,300.00",A,Official value
1,PP,Producer Prices,4,Afghanistan,5530,Producer Price (LCU/tonne),113,Rice,1994,1994,7021,Annual value,LCU,"6,200.00",A,Official value
2,PP,Producer Prices,4,Afghanistan,5530,Producer Price (LCU/tonne),113,Rice,1995,1995,7021,Annual value,LCU,"13,500.00",A,Official value
3,PP,Producer Prices,4,Afghanistan,5530,Producer Price (LCU/tonne),113,Rice,1996,1996,7021,Annual value,LCU,"18,000.00",A,Official value
4,PP,Producer Prices,4,Afghanistan,5530,Producer Price (LCU/tonne),113,Rice,1997,1997,7021,Annual value,LCU,"15,110.00",A,Official value


In [12]:
# Kiểm tra cấu trúc .info() và độ khuyết thiếu
df_price.info()
print("\nSố lượng giá trị null theo cột:")
print(df_price.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15334 entries, 0 to 15333
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       15334 non-null  object 
 1   Domain            15334 non-null  object 
 2   Area Code (M49)   15334 non-null  int64  
 3   Area              15334 non-null  object 
 4   Element Code      15334 non-null  int64  
 5   Element           15334 non-null  object 
 6   Item Code (CPC)   15334 non-null  int64  
 7   Item              15334 non-null  object 
 8   Year Code         15334 non-null  int64  
 9   Year              15334 non-null  int64  
 10  Months Code       15334 non-null  int64  
 11  Months            15334 non-null  object 
 12  Unit              12198 non-null  object 
 13  Value             15334 non-null  float64
 14  Flag              15334 non-null  object 
 15  Flag Description  15334 non-null  object 
dtypes: float64(1), int64(6), object(9)
memor

In [13]:
# Tìm hiểu tại sao cột Unit có giá trị null?
# Khám phá mối quan hệ giữa Element và Unit
df_price.groupby(['Element', 'Unit'], dropna=False).size().reset_index(name='Số dòng')


,Element,Unit,Số dòng
0,Producer Price (LCU/tonne),LCU,7654
1,Producer Price (SLC/tonne),SLC,2310
2,Producer Price (USD/tonne),USD,2234
3,Producer Price Index (2014-2016 = 100),NaN,3136


In [14]:
# Khám phá độ mịn thời gian: Năm và Tháng
print("Danh sách các chu kỳ thời gian (Months):")
print(df_price['Months'].value_counts())
print(f"Thời gian: Từ năm {df_price['Year'].min()} đến {df_price['Year'].max()}")


Danh sách các chu kỳ thời gian (Months):
Months
Annual value    9993
January          522
December         456
November         451
March            448
October          444
February         444
April            438
May              438
June             432
August           423
September        423
July             422
Name: count, dtype: int64
Thời gian: Từ năm 1991 đến 2025


In [15]:
# Xem giá lúa tại Việt Nam quy đổi ra USD/tấn qua các năm
df_vn_price_usd = df_price[
    (df_price['Area'] == 'Viet Nam') & 
    (df_price['Element'] == 'Producer Price (USD/tonne)') &
    (df_price['Months'] == 'Annual value')
]
print(f"Số năm có giá USD của Việt Nam: {len(df_vn_price_usd)}")
df_vn_price_usd[['Year', 'Value', 'Unit']].tail(10)


Số năm có giá USD của Việt Nam: 25


,Year,Value,Unit
15115,2015,287.10,USD
15116,2016,289.10,USD
15117,2017,284.60,USD
15118,2018,287.80,USD
15119,2019,281.00,USD
15120,2020,292.20,USD
15121,2021,329.60,USD
15122,2022,311.40,USD
15123,2023,326.70,USD
15124,2024,356.00,USD


### 1.3 Khám phá `supply_utilization.csv` (Cân đối Cung - Cầu Lúa Gạo)


In [16]:
# Đọc bảng cân đối Cung - Cầu (Supply Utilization Accounts)
f_su = DATA_DIR / "faostat" / "supply_utilization.csv"
df_su = pd.read_csv(f_su)
print(f"Kích thước bảng Cung - Cầu: {df_su.shape[0]:,} dòng x {df_su.shape[1]} cột")
df_su.head(5)


Kích thước bảng Cung - Cầu: 36,380 dòng x 15 cột


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,SCL,Supply Utilization Accounts (2010-),4,Afghanistan,5113,Opening stocks,113.00,Rice,2010,2010,t,"118,567.27",E,Estimated value,NaN
1,SCL,Supply Utilization Accounts (2010-),4,Afghanistan,5510,Production,113.00,Rice,2010,2010,t,"672,000.00",A,Official value,NaN
2,SCL,Supply Utilization Accounts (2010-),4,Afghanistan,5071,Stock Variation,113.00,Rice,2010,2010,t,"3,475.33",E,Estimated value,NaN
3,SCL,Supply Utilization Accounts (2010-),4,Afghanistan,5520,Feed,113.00,Rice,2010,2010,t,"2,097.31",I,Value imputed by a receiving agency,NaN
4,SCL,Supply Utilization Accounts (2010-),4,Afghanistan,5525,Seed,113.00,Rice,2010,2010,t,"20,775.91",I,Value imputed by a receiving agency,NaN


In [17]:
# Khám phá các thành phần cân đối cung cầu (Element)
print("Các thành tố cân đối cung cầu:")
print(df_su['Element'].value_counts())
print("\nMặt hàng:", df_su['Item'].unique())
print("Giai đoạn theo dõi:", df_su['Year'].min(), "đến", df_su['Year'].max())


Các thành tố cân đối cung cầu:
Element
Residuals                        4766
Opening stocks                   4607
Import quantity                  4586
Stock Variation                  4538
Export quantity                  3562
Production                       3287
Processed                        3041
Food supply quantity (tonnes)    2719
Feed                             1709
Loss                             1574
Seed                             1446
Other uses (non-food)             545
Name: count, dtype: int64

Mặt hàng: ['Rice' 'Rice, milled']
Giai đoạn theo dõi: 2010 đến 2023


In [18]:
# Xem bảng cân đối lúa gạo của Việt Nam trong năm gần nhất có đủ dữ liệu
latest_year = df_su['Year'].max()
vn_su_latest = df_su[(df_su['Area'] == 'Viet Nam') & (df_su['Year'] == latest_year)]
vn_su_latest[['Element', 'Item', 'Value', 'Unit', 'Flag Description']]


,Element,Item,Value,Unit,Flag Description
35643,Opening stocks,Rice,"9,998,415.60",t,Estimated value
35644,Production,Rice,"43,497,624.81",t,Official value
35645,Import quantity,Rice,"363,000.00",t,Estimated value
35646,Stock Variation,Rice,0,t,Estimated value
35647,Export quantity,Rice,517.45,t,Value from external organization
35648,Feed,Rice,"1,462,460.14",t,Estimated value
35649,Seed,Rice,"1,489,748.27",t,Value imputed by a receiving agency
35650,Loss,Rice,"3,862,073.12",t,Value imputed by a receiving agency
35651,Processed,Rice,"37,045,825.82",t,Estimated value
35652,Residuals,Rice,0,t,Value imputed by a receiving agency


### 1.4 Khám phá `Trade_DetailedTradeMatrix` (Ma trận Thương mại Gạo Song phương)


In [19]:
# Khám phá danh mục mã gạo trong file tham chiếu ItemCodes.csv
ref_items = DATA_DIR / "faostat" / "Trade_DetailedTradeMatrix_E_All_Data_(Normalized)" / "Trade_DetailedTradeMatrix_E_ItemCodes.csv"
df_items = pd.read_csv(ref_items, encoding='latin1')

# Lọc tìm các mã liên quan đến Rice/Paddy
rice_items = df_items[df_items['Item'].str.contains('rice|paddy', case=False, na=False)]
rice_items[['Item Code', 'CPC Code', 'Item']]


,Item Code,CPC Code,Item
42,35,'39120.02,Bran of rice
73,37,'21910.01,Cake of rice bran
133,110,'23490.01,Communion wafers; empty cachets of a kind suit...
202,38,'23120.01,Flour of rice
266,28,'23162,Husked rice
352,36,'21691.01,Oil of rice bran
460,27,'0113,Rice
461,32,'23161.03,Rice; broken
462,31,'23161.02,Rice; milled
463,29,'23161.01,Rice; milled (husked)


In [20]:
# Đọc thử mẫu 5 dòng từ ma trận thương mại gạo (rice_trade_data.csv)
f_trade = DATA_DIR / "faostat" / "Trade_DetailedTradeMatrix_E_All_Data_(Normalized)" / "rice_trade_data.csv"
sample_trade = pd.read_csv(f_trade, nrows=5)
print("Các cột có trong bảng ma trận thương mại gạo song phương:")
print(sample_trade.columns.tolist())
sample_trade.head()


Các cột có trong bảng ma trận thương mại gạo song phương:
['Reporter Country Code', 'Reporter Country Code (M49)', 'Reporter Countries', 'Partner Country Code', 'Partner Country Code (M49)', 'Partner Countries', 'Item Code', 'Item Code (CPC)', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag']


,Reporter Country Code,Reporter Country Code (M49),Reporter Countries,Partner Country Code,Partner Country Code (M49),Partner Countries,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag
0,7,'024,Angola,3,'008,Albania,28,'23162,Husked rice,5610,Import quantity,2022,2022,t,11.5,A
1,7,'024,Angola,3,'008,Albania,28,'23162,Husked rice,5622,Import value,2022,2022,1000 USD,5,A
2,7,'024,Angola,3,'008,Albania,30,'F0030,"Rice, paddy (rice milled equivalent)",5610,Import quantity,2022,2022,t,10.35,E
3,7,'024,Angola,3,'008,Albania,30,'F0030,"Rice, paddy (rice milled equivalent)",5622,Import value,2022,2022,1000 USD,5,E
4,20,'072,Botswana,2,'004,Afghanistan,110,'23490.01,"Communion wafers, empty cachets of a kind suit...",5610,Import quantity,2023,2023,t,0.12,A


In [21]:
# Thống kê nhanh toàn bộ file thương mại bằng phương pháp đọc từng phần (Chunking)
# Giữ an toàn bộ nhớ (OOM Safeguard)
total_trade_rows = 0
unique_reporters = set()
unique_partners = set()
flow_elements = set()
years_trade = set()

for chunk in pd.read_csv(f_trade, chunksize=200000, usecols=['Reporter Countries', 'Partner Countries', 'Element', 'Year', 'Value']):
    total_trade_rows += len(chunk)
    unique_reporters.update(chunk['Reporter Countries'].dropna().unique())
    unique_partners.update(chunk['Partner Countries'].dropna().unique())
    flow_elements.update(chunk['Element'].dropna().unique())
    years_trade.update(chunk['Year'].dropna().unique())

print(f"Tổng số dòng giao dịch song phương: {total_trade_rows:,} dòng")
print(f"Thời gian: Từ năm {min(years_trade)} đến {max(years_trade)}")
print(f"Số quốc gia xuất/báo cáo: {len(unique_reporters)} | Số đối tác thương mại: {len(unique_partners)}")
print("Các luồng thương mại:", flow_elements)


Tổng số dòng giao dịch song phương: 1,226,470 dòng
Thời gian: Từ năm 1986 đến 2024
Số quốc gia xuất/báo cáo: 195 | Số đối tác thương mại: 211
Các luồng thương mại: {'Import quantity', 'Export quantity', 'Export value', 'Import value'}


---
# 🇻🇳 PHẦN 2: DỮ LIỆU TỔNG CỤC THỐNG KÊ VIỆT NAM (NSO / GSO)
Khảo sát diện tích, năng suất, sản lượng lúa theo 63 tỉnh thành và 3 vụ mùa (Đông Xuân, Hè Thu/Thu Đông, Lúa Mùa).


### 2.1 Khám phá `metadata.txt` của GSO


In [22]:
# Đọc nội dung file mục lục metadata.txt
f_meta = DATA_DIR / "nso" / "metadata.txt"
with open(f_meta, "r", encoding="utf-8", errors="replace") as f:
    print(f.read())


12. Diện tích gieo trồng và sản lượng lúa cả năm

13. Diện tích lúa cả năm phân theo địa phương
14. Năng suất gieo trồng lúa cả năm phân theo địa phương
15. Sản lượng lúa cả năm phân theo địa phương

16. Diện tích gieo trồng lúa đông xuân phân theo địa phương
17. Năng suất gieo trồng lúa đông xuân phân theo địa phương
18. Sản lượng lúa đông xuân phân theo địa phương

19. Diện tích gieo trồng lúa hè thu và thu đông phân theo địa phương
20. Năng suất gieo trồng lúa hè thu và thu đông phân theo địa phương
21. Sản lượng lúa hè thu và thu đông phân theo địa phương

22. Diện tích gieo trồng lúa mùa phân theo địa phương
23. Năng suất gieo trồng lúa mùa phân theo địa phương
24. Sản lượng lúa mùa phân theo địa phương


### 2.2 Khám phá `V06.12.csv` (Chuỗi thời gian tổng hợp cả nước 1990–2024)


In [23]:
# Đọc V06.12.csv với mã ký tự CP1258 (hỗ trợ tiếng Việt GSO)
f_v12 = DATA_DIR / "nso" / "V06.12.csv"
df_v12 = pd.read_csv(f_v12, encoding='cp1258')
print("Kích thước:", df_v12.shape)
print("Tên các cột:")
for col in df_v12.columns:
    print(" -", col)
df_v12.head(5)


Kích thước: (71, 10)
Tên các cột:
 - Giá tr? và Ch? s? phát tri?n
 - Nam
 - T?ng di?n tích (Ngh́n ha)
 - Di?n tích Lúa  ?ông xuân (Ngh́n ha)
 - Di?n tích Lúa  hè thu (Ngh́n ha)
 - Di?n tích Lúa mùa (Ngh́n ha)
 - T?ng s?n lu?ng (Ngh́n t?n)
 - S?n lu?ng Lúa  ?ông xuân (Ngh́n t?n)
 - S?n lu?ng Lúa  hè thu (Ngh́n t?n)
 - S?n lu?ng Lúa mùa (Ngh́n t?n)


,Giá tr? và Ch? s? phát tri?n,Nam,T?ng di?n tích (Ngh́n ha),Di?n tích Lúa ?ông xuân (Ngh́n ha),Di?n tích Lúa hè thu (Ngh́n ha),Di?n tích Lúa mùa (Ngh́n ha),T?ng s?n lu?ng (Ngh́n t?n),S?n lu?ng Lúa ?ông xuân (Ngh́n t?n),S?n lu?ng Lúa hè thu (Ngh́n t?n),S?n lu?ng Lúa mùa (Ngh́n t?n)
0,Giá tr?,1990,"6,042.80","2,073.60","1,215.70","2,753.50","19,225.10","7,865.60","4,090.50","7,269.00"
1,Giá tr?,1991,"6,302.80","2,160.60","1,382.10","2,760.10","19,621.90","6,788.30","4,715.80","8,117.80"
2,Giá tr?,1992,"6,475.30","2,279.00","1,448.60","2,747.70","21,590.40","9,156.30","4,907.20","7,526.90"
3,Giá tr?,1993,"6,559.40","2,323.60","1,549.10","2,686.70","22,836.50","9,035.60","5,633.10","8,167.80"
4,Giá tr?,1994,"6,598.60","2,381.40","1,586.10","2,631.10","23,528.20","10,508.50","5,679.40","7,340.30"


In [24]:
# Phân tích 2 phần của bảng: 'Giá trị' (mức tuyệt đối) và 'Chỉ số phát triển' (tốc độ tăng trưởng)
print("Các chỉ tiêu phân loại ở cột đầu tiên:")
print(df_v12.iloc[:, 0].value_counts(dropna=False))

# Tách riêng bảng giá trị thực tế của diện tích & sản lượng cả nước
df_v12_values = df_v12[df_v12.iloc[:, 0] == 'Giá tr?'].copy()
print(f"Số năm có số liệu thực tế: {len(df_v12_values)} năm")
df_v12_values.tail(5)


Các chỉ tiêu phân loại ở cột đầu tiên:
Giá tr? và Ch? s? phát tri?n
Giá tr?                                   35
Ch? s? phát tri?n (Nam tru?c =100) - %    35
NaN                                        1
Name: count, dtype: int64
Số năm có số liệu thực tế: 35 năm


,Giá tr? và Ch? s? phát tri?n,Nam,T?ng di?n tích (Ngh́n ha),Di?n tích Lúa ?ông xuân (Ngh́n ha),Di?n tích Lúa hè thu (Ngh́n ha),Di?n tích Lúa mùa (Ngh́n ha),T?ng s?n lu?ng (Ngh́n t?n),S?n lu?ng Lúa ?ông xuân (Ngh́n t?n),S?n lu?ng Lúa hè thu (Ngh́n t?n),S?n lu?ng Lúa mùa (Ngh́n t?n)
30,Giá tr?,2020,"7,278.90","3,024.00","2,669.10","1,585.80","42,764.80","19,874.40","14,780.10","8,106.20"
31,Giá tr?,2021,"7,238.80","3,006.80","2,673.50","1,558.50","43,852.60","20,628.80","15,163.50","8,055.80"
32,Giá tr?,2022,"7,108.90","2,992.30","2,564.00","1,552.60","42,660.80","19,976.00","14,456.00","8,226.50"
33,Giá tr?,2023,"7,119.30","2,952.50","2,620.50","1,546.20","43,497.70","20,189.30","15,057.10","8,245.90"
34,Giá tr?,So b? 2024,"7,127.10","2,954.10","2,627.10","1,545.90","43,451.60","20,333.90","15,353.00","7,759.20"


### 2.3 Khám phá 12 bảng diện tích, năng suất, sản lượng cấp tỉnh (`V06.13` đến `V06.24`)


In [25]:
# Lập bảng tổng kết kích thước và đặc điểm cấu trúc của cả 12 file cấp tỉnh
nso_files = sorted(glob.glob(str(DATA_DIR / "nso" / "V06.*.csv")))
summary_list = []

for fpath in nso_files:
    fname = Path(fpath).name
    if fname == "V06.12.csv":
        continue
    df_temp = pd.read_csv(fpath, encoding='cp1258')
    year_cols = [c for c in df_temp.columns if any(ch.isdigit() for ch in str(c))]
    dots = (df_temp == '..').sum().sum()
    nans = df_temp.isna().sum().sum()
    summary_list.append({
        'File': fname,
        'Số dòng (Địa phương)': len(df_temp),
        'Số cột': len(df_temp.columns),
        'Cột năm': f"{year_cols[0]} - {year_cols[-1]}",
        'Ký hiệu thiếu ..': dots,
        'Ô NaN': nans
    })

pd.DataFrame(summary_list)


,File,Số dòng (Địa phương),Số cột,Cột năm,Ký hiệu thiếu ..,Ô NaN
0,V06.13.csv,72,31,1995 - So b? 2024,44,31
1,V06.14.csv,72,31,1995 - So b? 2024,44,31
2,V06.15.csv,72,31,1995 - So b? 2024,44,31
3,V06.16.csv,72,31,1995 - So b? 2024,57,31
4,V06.17.csv,72,31,1995 - So b? 2024,57,31
5,V06.18.csv,72,31,1995 - So b? 2024,57,31
6,V06.19.csv,38,31,1995 - So b? 2024,51,31
7,V06.20.csv,38,31,1995 - So b? 2024,54,31
8,V06.21.csv,38,31,1995 - So b? 2024,52,31
9,V06.22.csv,72,31,1995 - So b? 2024,162,31


**Phân tích phát hiện quan trọng:**
- Các file vụ **Hè Thu & Thu Đông (`V06.19`, `V06.20`, `V06.21`)** chỉ có **38 dòng**, trong khi các vụ khác có **72 dòng**.
- **Lý do nông học:** Vụ Hè Thu và Thu Đông chỉ được gieo cấy ở các tỉnh miền Trung và miền Nam (Bắc Trung Bộ, Duyên hải Nam Trung Bộ, Tây Nguyên, Đông Nam Bộ, Đồng bằng sông Cửu Long). Các tỉnh miền Bắc không canh tác vụ này mà làm vụ Mùa.


In [26]:
# Đọc sâu bảng Diện tích cả năm theo địa phương (V06.13.csv)
f_v13 = DATA_DIR / "nso" / "V06.13.csv"
df_v13 = pd.read_csv(f_v13, encoding='cp1258')
print("5 địa phương đầu tiên (gồm cả nước và vùng kinh tế):")
print(df_v13.iloc[:5, :6])

print("\n5 địa phương cuối bảng:")
print(df_v13.iloc[-5:, :6])


5 địa phương đầu tiên (gồm cả nước và vùng kinh tế):
       T?nh, thành ph?    1995    1996    1997    1998    1999
0              C? NU?C  6765.6  7003.8  7099.7  7362.7  7653.6
1  Đ?ng b?ng sông H?ng  1238.1  1216.6  1244.3  1250.3  1249.9
2               Hà N?i    56.1    54.8    54.5    54.0    53.9
3               Hà Tây   168.2   163.6   166.6   167.7   167.6
4            Vinh Phúc    72.1    67.9    71.4    74.1    71.8

5 địa phương cuối bảng:
   T?nh, thành ph?   1995   1996   1997   1998   1999
67       H?u Giang     ..     ..     ..     ..     ..
68       Sóc Trang  275.6  320.2  330.7  343.6  356.5
69        B?c Liêu  130.0  139.8  152.0  170.7  205.9
70          Cà Mau  187.1  201.1  220.9  234.0  243.8
71             NaN    NaN    NaN    NaN    NaN    NaN


In [27]:
# Khám phá top 5 tỉnh có sản lượng lúa lớn nhất Việt Nam trong năm gần nhất (V06.15.csv)
f_v15 = DATA_DIR / "nso" / "V06.15.csv"
df_v15 = pd.read_csv(f_v15, encoding='cp1258')

# Lấy cột năm gần nhất và chuyển kiểu số
latest_col = df_v15.columns[-1]
df_v15_clean = df_v15.copy()
df_v15_clean[latest_col] = pd.to_numeric(df_v15_clean[latest_col].astype(str).str.replace('..', 'NaN'), errors='coerce')

# Bỏ dòng tổng cả nước và các vùng tổng hợp để lấy riêng các tỉnh
provinces_only = df_v15_clean.dropna(subset=[latest_col]).sort_values(by=latest_col, ascending=False)
print(f"Top 10 đơn vị có sản lượng lúa cao nhất năm ({latest_col}):")
provinces_only.iloc[1:11][[df_v15.columns[0], latest_col]]


Top 10 đơn vị có sản lượng lúa cao nhất năm (So b? 2024):


,"T?nh, thành ph?",So b? 2024
57,Đ?ng b?ng sông C?u Long,"24,517.40"
29,B?c Trung B? và Duyên h?i mi?n Trung,"7,304.60"
1,Đ?ng b?ng sông H?ng,"5,345.90"
65,Kiên Giang,"4,714.60"
64,An Giang,"4,068.70"
63,Đ?ng Tháp,"3,331.20"
14,Trung du và mi?n núi phía B?c,"3,259.60"
58,Long An,"3,221.20"
68,Sóc Trang,"2,185.50"
44,Tây Nguyên,"1,567.10"


---
# 🌾 PHẦN 3: GIAO DỊCH GIÁ NỘI ĐỊA (THỊ TRƯỜNG NÔNG SẢN)
Khám phá `price_luagao.xlsx`: Giao dịch giá lúa tươi, gạo nguyên liệu, gạo thành phẩm hàng ngày.


In [28]:
# Đọc file Excel giá lúa gạo nội địa
f_ttns = DATA_DIR / "thitruongnongsan" / "price_luagao.xlsx"
df_ttns = pd.read_excel(f_ttns)
print(f"Kích thước bảng: {df_ttns.shape[0]:,} dòng x {df_ttns.shape[1]} cột")
df_ttns.head(5)


Kích thước bảng: 20,394 dòng x 8 cột


,Tên_mặt_hàng,Thị_trường,Loại_giá,Đơn_vị_tính,Loại_tiền,Nguồn,Ngày,Giá
0,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,8/3/2026 12:00:00 AM,"6,000.00"
1,Đài Loan,An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,7/31/2026 12:00:00 AM,"19,000.00"
2,Đài thơm 8,An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,7/31/2026 12:00:00 AM,"6,950.00"
3,Gạo Jasmine,An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,7/31/2026 12:00:00 AM,"15,500.00"
4,Gạo NL,An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,7/31/2026 12:00:00 AM,"9,570.00"


In [29]:
# Kiểm tra kiểu dữ liệu và giá trị thiếu của từng cột
df_ttns.info()
print("\nSố lượng giá trị khuyết thiếu:")
print(df_ttns.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20394 entries, 0 to 20393
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Tên_mặt_hàng  20394 non-null  object 
 1   Thị_trường    20394 non-null  object 
 2   Loại_giá      20394 non-null  object 
 3   Đơn_vị_tính   20394 non-null  object 
 4   Loại_tiền     20394 non-null  object 
 5   Nguồn         20394 non-null  object 
 6   Ngày          20394 non-null  object 
 7   Giá           20394 non-null  float64
dtypes: float64(1), object(7)
memory usage: 1.2+ MB

Số lượng giá trị khuyết thiếu:
Tên_mặt_hàng    0
Thị_trường      0
Loại_giá        0
Đơn_vị_tính     0
Loại_tiền       0
Nguồn           0
Ngày            0
Giá             0
dtype: int64


In [30]:
# Thống kê mô tả cột 'Giá' (VNĐ/kg)
print("Thống kê phân bố giá:")
print(df_ttns['Giá'].describe())
print(f"\nGiá nhỏ nhất: {df_ttns['Giá'].min():,.0f} VNĐ/kg")
print(f"Giá trung vị: {df_ttns['Giá'].median():,.0f} VNĐ/kg")
print(f"Giá lớn nhất: {df_ttns['Giá'].max():,.0f} VNĐ/kg")


Thống kê phân bố giá:
count   20,394.00
mean    10,704.75
std      5,513.99
min             0
25%      6,800.00
50%      8,741.67
75%     14,200.00
max     80,000.00
Name: Giá, dtype: float64

Giá nhỏ nhất: 0 VNĐ/kg
Giá trung vị: 8,742 VNĐ/kg
Giá lớn nhất: 80,000 VNĐ/kg


In [31]:
# Khám phá các mặt hàng (Tên_mặt_hàng): Đếm số lượng giống và top 15 giống giao dịch nhiều nhất
print(f"Tổng số chủng loại lúa/gạo được niêm yết: {df_ttns['Tên_mặt_hàng'].nunique()} giống")
print("\nTop 15 giống lúa gạo được cập nhật nhiều nhất:")
print(df_ttns['Tên_mặt_hàng'].value_counts().head(15))


Tổng số chủng loại lúa/gạo được niêm yết: 195 giống

Top 15 giống lúa gạo được cập nhật nhiều nhất:
Tên_mặt_hàng
Lúa IR 50404 (lúa tươi)    909
IR 50404                   719
Lúa IR 50404               676
OM 18                      639
OM 5451                    519
Jasmine                    497
Lúa OM 5451                405
Đài thơm 8                 385
Đài thơm 8 (lúa tươi)      349
Gạo thường                 315
Lúa nếp 97                 304
Lúa BC15                   304
RVT                        293
OM 6976                    284
Lúa OM 6976                282
Name: count, dtype: int64


In [32]:
# Khám phá các thị trường giao dịch (Thị_trường) và đơn vị tính
print("Các thị trường thu mua:")
print(df_ttns['Thị_trường'].value_counts())

print("\nCác loại đơn vị tính được ghi nhận:")
print(df_ttns['Đơn_vị_tính'].value_counts())


Các thị trường thu mua:
Thị_trường
An Giang      4165
Kiên Giang    3724
Cần Thơ       3082
Thái Bình     2720
Đồng Tháp     2469
Sóc Trăng     1101
Hậu Giang      708
Vĩnh Long      559
Trà Vinh       410
Hà Nội         366
Tiền Giang     307
Bến Tre        299
Cà Mau         275
Bạc Liêu        91
Long An         56
Tây Ninh        50
Hậu GIang       12
Name: count, dtype: int64

Các loại đơn vị tính được ghi nhận:
Đơn_vị_tính
VNĐ/Kg     10064
Vnđ/Kg      5072
VNĐ/kg      3880
Đồng/kg     1378
Name: count, dtype: int64


In [33]:
# Khám phá chuỗi thời gian: Chuyển đổi và tìm ngày bắt đầu, ngày kết thúc
df_ttns['Ngày_dt'] = pd.to_datetime(df_ttns['Ngày'], errors='coerce')
print(f"Giao dịch từ ngày: {df_ttns['Ngày_dt'].min().strftime('%Y-%m-%d')} đến ngày: {df_ttns['Ngày_dt'].max().strftime('%Y-%m-%d')}")
print(f"Số ngày giao dịch khác nhau: {df_ttns['Ngày_dt'].dt.date.nunique()} ngày")


Giao dịch từ ngày: 2021-01-04 đến ngày: 2026-08-03
Số ngày giao dịch khác nhau: 1212 ngày


C:\Users\hoang\AppData\Local\Temp\ipykernel_20836\493085248.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_ttns['Ngày_dt'] = pd.to_datetime(df_ttns['Ngày'], errors='coerce')


In [34]:
# KIỂM TOÁN CHẤT LƯỢNG: Phát hiện các bản ghi có Giá = 0 bất thường
zero_price_rows = df_ttns[df_ttns['Giá'] <= 0]
print(f"Số lượng bản ghi có giá bằng 0: {len(zero_price_rows)} bản ghi")
zero_price_rows[['Tên_mặt_hàng', 'Thị_trường', 'Ngày', 'Giá']].head(10)


Số lượng bản ghi có giá bằng 0: 29 bản ghi


,Tên_mặt_hàng,Thị_trường,Ngày,Giá
12243,Gạo NL 5% tấm,Kiên Giang,6/9/2023 12:00:00 AM,0
12275,Gạo NL 5% tấm,Kiên Giang,6/7/2023 12:00:00 AM,0
12385,Gạo NL 5% tấm,Kiên Giang,5/26/2023 12:00:00 AM,0
12417,Gạo NL 5% tấm,Kiên Giang,5/24/2023 12:00:00 AM,0
12492,Gạo NL 5% tấm,Kiên Giang,5/19/2023 12:00:00 AM,0
12541,Gạo NL 5% tấm,Kiên Giang,5/17/2023 12:00:00 AM,0
12637,Gạo NL 5% tấm,Kiên Giang,5/12/2023 12:00:00 AM,0
12728,Gạo NL 5% tấm,Kiên Giang,5/10/2023 12:00:00 AM,0
12833,Gạo NL 5% tấm,Kiên Giang,5/5/2023 12:00:00 AM,0
12902,Gạo NL 5% tấm,Kiên Giang,5/3/2023 12:00:00 AM,0


---
# 🇺🇸 PHẦN 4: BỘ NÔNG NGHIỆP HOA KỲ (USDA FAS)
Khám phá:
1. `Export-prices-Thailand-Vietnam-India-and-Pakistan.csv`: Giá chào xuất khẩu FOB gạo hàng tháng.
2. `usda.xls`: Bảng cân đối vĩ mô Cung - Cầu PSD (Production, Supply and Distribution) của Việt Nam.


### 4.1 Khám phá `Export-prices-...csv` (Giá xuất khẩu 4 nước lớn)


In [35]:
# Đọc dữ liệu giá xuất khẩu gạo USDA
f_usda_exp = DATA_DIR / "usda" / "Export-prices-Thailand-Vietnam-India-and-Pakistan.csv"
df_usda_exp = pd.read_csv(f_usda_exp)
print(f"Kích thước bảng giá xuất khẩu: {df_usda_exp.shape[0]:,} dòng x {df_usda_exp.shape[1]} cột")
df_usda_exp.head(5)


Kích thước bảng giá xuất khẩu: 13,131 dòng x 10 cột


,TABLE_NAME,TABLE_NUMBER,COMMODITY_DESCRIPTION,CLASS_DESCRIPTION,YEAR,REFERENCE_PERIOD_DESCRIPTION,LOCATION_DESCRIPTION,STATISTIC_DESCRIPTION,VALUE,UNIT_DESCRIPTION
0,"Thailand milled rice prices, f.o.b. Bangkok, 1...",25,RICE,FRAGRANT WHITE RICE,1985,August,THAILAND,FREIGHT ON BOARD PRICE,NaN,DOLLARS PER METRIC TON
1,"Thailand milled rice prices, f.o.b. Bangkok, 1...",25,RICE,100-PERCENT GRADE B,1985,August,THAILAND,FREIGHT ON BOARD PRICE,193.38,DOLLARS PER METRIC TON
2,"Thailand milled rice prices, f.o.b. Bangkok, 1...",25,RICE,5-PERCENT PARBOILED,1985,August,THAILAND,FREIGHT ON BOARD PRICE,178.50,DOLLARS PER METRIC TON
3,"Thailand milled rice prices, f.o.b. Bangkok, 1...",25,RICE,5-PERCENT BROKEN,1985,August,THAILAND,FREIGHT ON BOARD PRICE,NaN,DOLLARS PER METRIC TON
4,"Thailand milled rice prices, f.o.b. Bangkok, 1...",25,RICE,15-PERCENT BROKEN,1985,August,THAILAND,FREIGHT ON BOARD PRICE,NaN,DOLLARS PER METRIC TON


In [36]:
# Kiểm tra kiểu dữ liệu và tỷ lệ thiếu cột VALUE
df_usda_exp.info()
print("\nTỷ lệ thiếu theo cột:")
print((df_usda_exp.isna().sum() / len(df_usda_exp) * 100).round(2).astype(str) + '%')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13131 entries, 0 to 13130
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   TABLE_NAME                    13131 non-null  object 
 1   TABLE_NUMBER                  13131 non-null  int64  
 2   COMMODITY_DESCRIPTION         13131 non-null  object 
 3   CLASS_DESCRIPTION             13131 non-null  object 
 4   YEAR                          13131 non-null  int64  
 5   REFERENCE_PERIOD_DESCRIPTION  13131 non-null  object 
 6   LOCATION_DESCRIPTION          13131 non-null  object 
 7   STATISTIC_DESCRIPTION         13131 non-null  object 
 8   VALUE                         8537 non-null   float64
 9   UNIT_DESCRIPTION              13131 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 1.0+ MB

Tỷ lệ thiếu theo cột:
TABLE_NAME                        0.0%
TABLE_NUMBER                      0.0%
COMMODITY_DESCRIPTIO

In [37]:
# Khám phá các quốc gia và phẩm cấp gạo tương ứng
print("Số dòng theo từng quốc gia:")
print(df_usda_exp['LOCATION_DESCRIPTION'].value_counts())

print("\nCác phẩm cấp gạo của VIỆT NAM:")
print(df_usda_exp[df_usda_exp['LOCATION_DESCRIPTION'] == 'VIETNAM']['CLASS_DESCRIPTION'].unique())

print("\nCác phẩm cấp gạo của THÁI LAN:")
print(df_usda_exp[df_usda_exp['LOCATION_DESCRIPTION'] == 'THAILAND']['CLASS_DESCRIPTION'].unique())


Số dòng theo từng quốc gia:
LOCATION_DESCRIPTION
THAILAND    4752
PAKISTAN    3080
INDIA       2695
VIETNAM     2604
Name: count, dtype: int64

Các phẩm cấp gạo của VIỆT NAM:
['5-PERCENT DOUBLE WATER POLISHED' '5-PERCENT BROKEN' '10-PERCENT BROKEN'
 '15-PERCENT BROKEN' '25-PERCENT BROKEN' '100-PERCENT BROKEN'
 'JASMINE (AROMATIC)']

Các phẩm cấp gạo của THÁI LAN:
['FRAGRANT WHITE RICE' '100-PERCENT GRADE B' '5-PERCENT PARBOILED'
 '5-PERCENT BROKEN' '15-PERCENT BROKEN' '25-PERCENT BROKEN'
 '35-PERCENT BROKEN' 'A.1 SUPER' 'A.1 SPECIAL']


In [38]:
# So sánh tỷ lệ thiếu giá giữa 4 quốc gia xuất khẩu
for loc, grp in df_usda_exp.groupby('LOCATION_DESCRIPTION'):
    missing_cnt = grp['VALUE'].isna().sum()
    total_cnt = len(grp)
    print(f"- {loc:10}: Thiếu {missing_cnt:4,}/{total_cnt:4,} giá ({missing_cnt/total_cnt*100:.1f}%) | Năm: {grp['YEAR'].min()} - {grp['YEAR'].max()}")


- INDIA     : Thiếu 1,294/2,695 giá (48.0%) | Năm: 1997 - 2025
- PAKISTAN  : Thiếu 1,138/3,080 giá (36.9%) | Năm: 1997 - 2025
- THAILAND  : Thiếu 1,213/4,752 giá (25.5%) | Năm: 1985 - 2025
- VIETNAM   : Thiếu  949/2,604 giá (36.4%) | Năm: 1997 - 2025


In [39]:
# Thống kê mức giá xuất khẩu gạo 5% tấm (5-PERCENT BROKEN) giữa Việt Nam và Thái Lan
g5_df = df_usda_exp[df_usda_exp['CLASS_DESCRIPTION'] == '5-PERCENT BROKEN']
g5_df.groupby('LOCATION_DESCRIPTION')['VALUE'].describe()


,count,mean,std,min,25%,50%,75%,max
LOCATION_DESCRIPTION,,,,,,,,
INDIA,262.00,349.13,80.27,168.00,276.25,370.50,408.75,495.00
PAKISTAN,318.00,381.33,109.93,169.00,270.00,390.00,441.75,850.00
THAILAND,517.00,358.11,128.35,162.60,267.48,331.00,430.00,930.00
VIETNAM,356.00,365.62,131.27,148.00,255.00,379.00,435.25,"1,075.00"


### 4.2 Khám phá `usda.xls` (Bảng cân đối PSD lúa gạo Việt Nam)


In [40]:
# Đọc file usda.xls bằng parser HTML (do file thực chất là bảng HTML lưu đuôi .xls)
f_usda_xls = DATA_DIR / "usda" / "usda.xls"
tables = pd.read_html(f_usda_xls)
df_psd = tables[0]
print(f"Kích thước bảng PSD: {df_psd.shape[0]} chỉ tiêu x {df_psd.shape[1]} cột niên vụ")
df_psd.iloc[:5, :6]


Kích thước bảng PSD: 15 chỉ tiêu x 70 cột niên vụ


,Commodity,Attribute,Country,1960/1961,1961/1962,1962/1963
0,"Rice, Milled",Area Harvested,Vietnam,"4,602.00","4,763.00","4,889.00"
1,NaN,Beginning Stocks,Vietnam,0,0,0
2,NaN,Production,Vietnam,"5,959.00","6,018.00","6,335.00"
3,NaN,Rough Production,Vietnam,"9,168.00","9,258.00","9,746.00"
4,NaN,Milling Rate (.9999),Vietnam,"6,500.00","6,500.00","6,500.00"


In [41]:
# Danh sách 15 chỉ tiêu cân đối vĩ mô (Attribute) của USDA cho ngành lúa gạo Việt Nam
psd_attributes = df_psd[['Attribute', 'Unit Description']].copy()
psd_attributes


,Attribute,Unit Description
0,Area Harvested,(1000 HA)
1,Beginning Stocks,(1000 MT)
2,Production,(1000 MT)
3,Rough Production,(1000 MT)
4,Milling Rate (.9999),(1000 MT)
5,Imports,(1000 MT)
6,TY Imports,(1000 MT)
7,TY Imp. from U.S.,(1000 MT)
8,Total Supply,(1000 MT)
9,Exports,(1000 MT)


In [42]:
# Xem sản lượng và xuất khẩu gạo của Việt Nam trong 5 niên vụ gần nhất
crop_years = [c for c in df_psd.columns if '/' in str(c)]
recent_years = crop_years[-5:]

psd_recent = df_psd[df_psd['Attribute'].isin(['Production', 'Exports', 'Domestic Consumption', 'Ending Stocks'])]
psd_recent[['Attribute'] + recent_years]


,Attribute,2021/2022,2022/2023,2023/2024,2024/2025,2025/2026
2,Production,"26,670.00","27,140.00","27,200.00","26,850.00","26,200.00"
9,Exports,"7,054.00","8,225.00","9,035.00","8,063.00","8,100.00"
11,Domestic Consumption,"22,050.00","22,250.00","22,300.00","22,300.00","22,250.00"
12,Ending Stocks,"3,605.00","3,770.00","3,335.00","3,347.00","2,297.00"


---
# 🌐 PHẦN 5: NGÂN HÀNG THẾ GIỚI (WORLD BANK PINK SHEET)
Khám phá `CMO-Historical-Data-Monthly.xlsx`: Chuỗi giá hàng hóa toàn cầu hàng tháng (1960–2025).


In [43]:
# Khám phá các sheet có trong file Excel của World Bank
f_wb = DATA_DIR / "world_bank" / "CMO-Historical-Data-Monthly.xlsx"
wb_obj = openpyxl.load_workbook(f_wb, read_only=True)
print("Các sheet có trong file:", wb_obj.sheetnames)
wb_obj.close()


Các sheet có trong file: ['AFOSHEET', 'Monthly Prices', 'Monthly Indices', 'Description', 'Index Weights']


In [44]:
# Đọc sheet 'Monthly Prices' với việc bỏ qua 4 dòng tiêu đề đầu tiên
df_wb_raw = pd.read_excel(f_wb, sheet_name='Monthly Prices', skiprows=4)
print("Kích thước bảng:", df_wb_raw.shape)

# Cột đầu tiên là Tháng (Date), dòng đầu tiên chứa Đơn vị (Unit)
df_wb_raw.rename(columns={df_wb_raw.columns[0]: 'Date'}, inplace=True)
print("5 dòng đầu tiên:")
df_wb_raw.iloc[:5, :8]


Kích thước bảng: (793, 72)
5 dòng đầu tiên:


,Date,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US"
0,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu)
1,1960M01,1.63,1.63,1.63,…,…,…,0.14
2,1960M02,1.63,1.63,1.63,…,…,…,0.14
3,1960M03,1.63,1.63,1.63,…,…,…,0.14
4,1960M04,1.63,1.63,1.63,…,…,…,0.14


In [45]:
# Tách dòng đơn vị và làm sạch tên các cột
units_row = df_wb_raw.iloc[0]
df_wb = df_wb_raw.iloc[1:].copy()
df_wb.columns = [str(c).strip() for c in df_wb.columns]

# Thay thế các ký tự thiếu của World Bank (dấu ba chấm '…' và '..') thành NaN
df_wb.replace({'…': np.nan, '..': np.nan, '': np.nan}, inplace=True)

print(f"Tổng số tháng được ghi nhận: {len(df_wb)} tháng")
print(f"Từ tháng {df_wb['Date'].iloc[0]} đến tháng {df_wb['Date'].iloc[-1]}")


Tổng số tháng được ghi nhận: 792 tháng
Từ tháng 1960M01 đến tháng 2025M12


C:\Users\hoang\AppData\Local\Temp\ipykernel_20836\79292761.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_wb.replace({'…': np.nan, '..': np.nan, '': np.nan}, inplace=True)


In [46]:
# Lọc và khảo sát riêng các biến mục tiêu Lúa Gạo & Biến vĩ mô đầu vào (Urea, DAP, Dầu thô)
target_cols = [
    'Rice, Viet Namese 5%',
    'Rice, Thai 5%',
    'Rice, Thai 25%',
    'Urea',
    'DAP',
    'Crude oil, Brent',
    'Crude oil, average'
]

# Chuyển đổi sang kiểu số
for c in target_cols:
    df_wb[c] = pd.to_numeric(df_wb[c], errors='coerce')

# Thống kê mô tả
df_wb[target_cols].describe()


,"Rice, Viet Namese 5%","Rice, Thai 5%","Rice, Thai 25%",Urea,DAP,"Crude oil, Brent","Crude oil, average"
count,254.00,792.00,472.00,792.00,708.00,792.00,792.00
mean,397.78,308.18,328.20,173.56,261.09,34.76,33.93
std,94.68,142.41,126.52,142.25,184.70,32.56,31.4
min,197.00,95.67,83.26,16,54,1.21,1.21
25%,348.76,192.50,231.00,77.3,146.00,11.51,11.51
50%,392.41,281.45,292.38,130.00,186.31,23.65,23.35
75%,444.53,410.00,421.00,242.50,357.28,55.81,54.34
max,780.00,907.00,700.00,925.00,"1,075.75",133.87,132.83


In [47]:
# So sánh thời điểm bắt đầu có dữ liệu của Gạo Việt Nam vs Gạo Thái Lan vs Dầu Thô
for c in target_cols:
    valid_series = df_wb.dropna(subset=[c])
    start_m = valid_series['Date'].iloc[0] if len(valid_series) > 0 else 'N/A'
    end_m = valid_series['Date'].iloc[-1] if len(valid_series) > 0 else 'N/A'
    pct = len(valid_series) / len(df_wb) * 100
    print(f"- {c:22}: Có {len(valid_series):3} tháng ({pct:5.1f}%) | Bắt đầu: {start_m} -> {end_m}")


- Rice, Viet Namese 5%  : Có 254 tháng ( 32.1%) | Bắt đầu: 2003M12 -> 2025M12
- Rice, Thai 5%         : Có 792 tháng (100.0%) | Bắt đầu: 1960M01 -> 2025M12
- Rice, Thai 25%        : Có 472 tháng ( 59.6%) | Bắt đầu: 1986M01 -> 2025M12
- Urea                  : Có 792 tháng (100.0%) | Bắt đầu: 1960M01 -> 2025M12
- DAP                   : Có 708 tháng ( 89.4%) | Bắt đầu: 1967M01 -> 2025M12
- Crude oil, Brent      : Có 792 tháng (100.0%) | Bắt đầu: 1960M01 -> 2025M12
- Crude oil, average    : Có 792 tháng (100.0%) | Bắt đầu: 1960M01 -> 2025M12


In [48]:
# Ma trận tương quan (Correlation Matrix) giữa Giá Gạo và Giá Phân bón / Dầu thô
# Tính trên giai đoạn từ khi Việt Nam có dữ liệu (2004 đến nay)
df_wb_recent = df_wb[df_wb['Date'] >= '2004M01'][target_cols]
corr_matrix = df_wb_recent.corr().round(2)
print("Ma trận hệ số tương quan tuyến tính Pearson (2004 - 2025):")
corr_matrix


Ma trận hệ số tương quan tuyến tính Pearson (2004 - 2025):


,"Rice, Viet Namese 5%","Rice, Thai 5%","Rice, Thai 25%",Urea,DAP,"Crude oil, Brent","Crude oil, average"
"Rice, Viet Namese 5%",1,0.9,0.91,0.43,0.61,0.52,0.52
"Rice, Thai 5%",0.9,1,0.98,0.4,0.56,0.53,0.53
"Rice, Thai 25%",0.91,0.98,1,0.41,0.54,0.51,0.5
Urea,0.43,0.4,0.41,1,0.86,0.6,0.61
DAP,0.61,0.56,0.54,0.86,1,0.58,0.6
"Crude oil, Brent",0.52,0.53,0.51,0.6,0.58,1,1
"Crude oil, average",0.52,0.53,0.5,0.61,0.6,1,1


---
## 🎯 KẾT LUẬN TỔNG HỢP SAU QUÁ TRÌNH EDA TỪNG BƯỚC
1. **FAOSTAT:**
   - Cung cấp bức tranh toàn cảnh 1961–2024. Số liệu sản lượng lúa của Việt Nam đầy đủ 100%.
   - Đơn vị năng suất của FAO là `hg/ha` (1 hg = 100g = 0.1 kg), cần quy đổi về tấn/ha hoặc tạ/ha.
2. **NSO Việt Nam:**
   - 12 file cấp tỉnh ở dạng ma trận rộng (các năm là cột: 1995 – 2024). Cần unpivot/melt khi đưa vào Data Lakehouse.
   - Vụ Hè Thu & Thu Đông chỉ có 38 địa phương do yếu tố khí hậu miền Bắc không trồng vụ này.
   - Cần thay thế ký hiệu `..` thành `NULL`.
3. **Thị trường nông sản nội địa:**
   - 20,394 bản ghi giá hàng ngày, bao phủ 195 giống lúa.
   - Phát hiện đúng 29 bản ghi có Giá = 0 (Gạo NL 5% tấm tại Kiên Giang tháng 5-6/2023).
4. **USDA FAS:**
   - Tỷ lệ thiếu giá xuất khẩu tháng là 34.98%, chủ yếu do lệnh cấm xuất khẩu (như Ấn Độ) hoặc thời điểm ngoài vụ.
   - Bảng PSD của Việt Nam có 15 chỉ tiêu qua 66 niên vụ (1960/1961 đến 2025/2026), là bảng cân đối cung cầu chính thức chuẩn hóa nhất.
5. **World Bank:**
   - Gạo Việt Nam 5% bắt đầu ghi nhận từ tháng 12/2003 (`2003M12`).
   - Tương quan rất cao giữa giá Dầu thô / Phân bón Urea với biến động giá gạo xuất khẩu thế giới.
